In [ ]:
!apt-get update -qq
!apt-get install -y -qq libreoffice-core libreoffice-writer
!pip install -q PyPDF2 python-docx openpyxl streamlit
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb

!dpkg -i cloudflared-linux-amd64.deb

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ============================================
# CONFIGURAÇÃO DOS CAMINHOS DO PROJETO
# ============================================
# Esta seção "junta" todos os diretórios utilizados
# pela aplicação. Dessa forma, qualquer alteração na
# estrutura de pastas pode ser realizada em apenas um
# local do código, facilitando a manutenção do sistema.
# ============================================

import os

PASTA_BASE = ""
PASTA_BACKEND = f"{PASTA_BASE}/backend"
PASTA_ENTRADA = f"{PASTA_BASE}/entrada"
PASTA_MODELOS = f"{PASTA_BASE}/modelos"
PASTA_SAIDA = f"{PASTA_BASE}/saida"

os.makedirs(PASTA_BACKEND, exist_ok=True)
os.makedirs(PASTA_ENTRADA, exist_ok=True)
os.makedirs(PASTA_MODELOS, exist_ok=True)
os.makedirs(PASTA_SAIDA, exist_ok=True)

print("Estrutura preparada com sucesso!")
print(PASTA_BASE)

Estrutura preparada com sucesso!
/content/drive/MyDrive/Colab_Projetos/Gerador_Documentos


In [ ]:
import os

caminho_backend = f"{PASTA_BACKEND}/backend_gerador.py"

if os.path.exists(caminho_backend):
    print("Backend encontrado com sucesso:")
    print(caminho_backend)
else:
    print("Backend NÃO encontrado. Verifique se o arquivo backend_gerador.py foi criado.")

Backend encontrado com sucesso:
/content/drive/MyDrive/Colab_Projetos/Gerador_Documentos/backend/backend_gerador.py


In [ ]:
%%writefile /content/app_streamlit.py
import os
import sys
import tempfile
import streamlit as st

# ============================================
# CAMINHOS DO PROJETO
# ============================================

PASTA_BASE = "/content/drive/MyDrive/Colab_Projetos/Gerador_Documentos"
PASTA_BACKEND = f"{PASTA_BASE}/backend"
PASTA_MODELOS_PADRAO = f"{PASTA_BASE}/modelos"
PASTA_SAIDA_PADRAO = f"{PASTA_BASE}/saida"

if PASTA_BACKEND not in sys.path:
    sys.path.append(PASTA_BACKEND)

from backend_gerador import (
    processar_documentos,
    localizar_excel_unico,
    localizar_modelos
)

# ============================================
# CONFIGURAÇÃO DA PÁGINA
# ============================================

st.set_page_config(
    page_title="Gerador de Documentos",
    page_icon="/content/drive/MyDrive/Colab_Projetos/Gerador_Documentos/Image_favicon.png",
    layout="wide",
    initial_sidebar_state="expanded"
)

# ============================================
# FUNÇÃO AUXILIAR: SALVAR EXCEL ENVIADO
# ============================================

def preparar_pasta_temporaria_excel(uploaded_file):

    if uploaded_file is None:
        raise ValueError("Nenhum arquivo Excel foi enviado.")

    pasta_temp = os.path.join(
        tempfile.gettempdir(),
        "entrada_streamlit_excel"
    )

    os.makedirs(pasta_temp, exist_ok=True)

    for nome in os.listdir(pasta_temp):
        if nome.lower().endswith(".xlsx"):
            os.remove(os.path.join(pasta_temp, nome))

    caminho_excel = os.path.join(
        pasta_temp,
        "entrada.xlsx"
    )

    with open(caminho_excel, "wb") as f:
        f.write(uploaded_file.getbuffer())

    return pasta_temp, caminho_excel

def preparar_modelos_temporarios(uploaded_adv, uploaded_susp):

    pasta_temp = os.path.join(
        tempfile.gettempdir(),
        "modelos_streamlit"
    )

    os.makedirs(pasta_temp, exist_ok=True)

    for arquivo in os.listdir(pasta_temp):
        if arquivo.lower().endswith(".docx"):
            os.remove(os.path.join(pasta_temp, arquivo))

    if uploaded_adv is None:
        raise ValueError(
            "Envie o modelo de advertência."
        )

    if uploaded_susp is None:
        raise ValueError(
            "Envie o modelo de suspensão."
        )

    with open(
        os.path.join(
            pasta_temp,
            "MODELO_ADVERTENCIA.docx"
        ),
        "wb"
    ) as f:

        f.write(uploaded_adv.getbuffer())

    with open(
        os.path.join(
            pasta_temp,
            "MODELO_SUSPENSAO.docx"
        ),
        "wb"
    ) as f:

        f.write(uploaded_susp.getbuffer())

    return pasta_temp

# ============================================
# ESTADO DA SESSÃO
# ============================================

if "resultado" not in st.session_state:
    st.session_state.resultado = None

if "validacao" not in st.session_state:
    st.session_state.validacao = None

# ============================================
# CABEÇALHO VERSÃO
# ============================================

st.title("Gerador de Suspensões e Advertências")
st.caption("Versão 1.3")

# ============================================
# SIDEBAR
# ============================================

with st.sidebar:
    st.header("Configurações")

    uploaded_excel = st.file_uploader(
    "Enviar planilha Excel (.xlsx)",
    type=["xlsx"]
    )

    origem_modelos = st.radio(
        "Origem dos modelos",
        [
            "Utilizar modelos padrão do sistema",
            "Enviar modelos personalizados"
        ]
    )

if origem_modelos == "Utilizar modelos padrão do sistema":

    pasta_modelos = st.text_area(
        "Pasta de modelos",
        value=PASTA_MODELOS_PADRAO,
        height=68
    )

    uploaded_adv = None
    uploaded_susp = None

else:

    uploaded_adv = st.file_uploader(
        "Modelo de Advertência (.docx)",
        type=["docx"],
        key="adv"
    )

    uploaded_susp = st.file_uploader(
        "Modelo de Suspensão (.docx)",
        type=["docx"],
        key="susp"
    )

    pasta_modelos = None



pasta_saida = st.text_area(
    "Pasta de saída",
    value=PASTA_SAIDA_PADRAO,
    height=68
)

opcao_legivel = st.radio(
    "Tipo de processamento",
    options=[
        "Ambos",
        "Somente Suspensões",
        "Somente Advertências"
    ]
)

mapa_opcao = {
    "Ambos": "ambos",
    "Somente Suspensões": "suspensoes",
    "Somente Advertências": "advertencias"
}

opcao = mapa_opcao[opcao_legivel]

copias = st.selectbox(
    "Cópias por página",
    options=[1, 2, 3, 4, 5],
    index=1
)

st.divider()

col1, col2 = st.columns(2)

with col1:
    btn_validar = st.button("🔎 Validar")

with col2:
    btn_limpar = st.button("🧹 Limpar")

pode_processar = True

if uploaded_excel is None:
    pode_processar = False

if (
    origem_modelos == "Enviar modelos personalizados"
    and (
        uploaded_adv is None
        or uploaded_susp is None
    )
):
    pode_processar = False

btn_processar = st.button(
    "🚀 Gerar documentos",
    type="primary",
    use_container_width=True,
    disabled=not pode_processar
)

# ============================================
# AÇÕES
# ============================================

if btn_limpar:
    st.session_state.resultado = None
    st.session_state.validacao = None
    st.rerun()


if btn_validar:

    try:

        pasta_entrada_temp, _ = (
            preparar_pasta_temporaria_excel(uploaded_excel)
        )

        excel_localizado = localizar_excel_unico(
            pasta_entrada_temp
        )

        if origem_modelos == "Utilizar modelos padrão do sistema":

            st.success("📂 Utilizando modelos salvos no Drive.")

            pasta_modelos_validacao = (
                pasta_modelos.strip()
            )

        else:

            st.info("📤 Utilizando modelos enviados pelo navegador.")

            pasta_modelos_validacao = (
                preparar_modelos_temporarios(
                    uploaded_adv,
                    uploaded_susp
                )
            )

        modelo_advertencia, modelo_suspensao = (
            localizar_modelos(
                pasta_modelos_validacao
            )
        )

        st.session_state.validacao = {
            "excel": excel_localizado,
            "excel_nome_original": uploaded_excel.name,
            "advertencia": modelo_advertencia,
            "suspensao": modelo_suspensao
        }

    except Exception as e:

        st.session_state.validacao = {
            "erro": str(e)
        }

        st.error("Erro na validação.")


if btn_processar:

    progresso = st.progress(0)
    status = st.empty()

    try:

        status.info("Preparando arquivos...")
        progresso.progress(10)

        pasta_entrada_temp, _ = (
            preparar_pasta_temporaria_excel(uploaded_excel)
        )

        progresso.progress(30)

        if origem_modelos == "Enviar modelos personalizados":

            status.info("Preparando modelos enviados...")

            pasta_modelos_processamento = (
                preparar_modelos_temporarios(
                    uploaded_adv,
                    uploaded_susp
                )
            )

        else:

            pasta_modelos_processamento = (
                pasta_modelos.strip()
            )

        progresso.progress(50)
        status.info("Gerando documentos...")

        resultado = processar_documentos(
            pasta_entrada=pasta_entrada_temp,
            pasta_modelos=pasta_modelos_processamento,
            pasta_saida=pasta_saida.strip(),
            opcao=opcao,
            copias=int(copias)
        )

        progresso.progress(90)
        status.info("Finalizando...")

        if uploaded_excel is not None:
            resultado["excel_nome_original"] = uploaded_excel.name

        st.session_state.resultado = resultado

        progresso.progress(100)
        status.success("Processamento concluído!")

    except Exception as e:

        st.session_state.resultado = {
            "erro": str(e)
        }

        status.error("Erro durante o processamento.")

# ============================================
# ABAS
# ============================================

aba1, aba2, aba3 = st.tabs(["📌 Visão geral", "📂 Arquivos detectados", "📝 Relatório"])

# ============================================
# ABA 1 — VISÃO GERAL
# ============================================

with aba1:
    st.subheader("Visão geral")

    if st.session_state.resultado is None and st.session_state.validacao is None:
        st.info("Envie a planilha Excel na barra lateral, ajuste os caminhos se necessário e clique em Validar ou Gerar documentos.")

    elif st.session_state.resultado and "erro" in st.session_state.resultado:
        st.error(st.session_state.resultado["erro"])

    elif st.session_state.resultado:
        r = st.session_state.resultado

        st.success("Processamento concluído com sucesso!")

        c1, c2, c3, c4 = st.columns(4)

        with c1:
            st.markdown("**Linhas no Excel**")
            st.write(r["linhas_excel"])

        with c2:
            st.markdown("**Advertências**")
            st.write(r["total_advertencias"])

        with c3:
            st.markdown("**Suspensões**")
            st.write(r["total_suspensoes"])

        with c4:
            st.markdown("**Sem data**")
            st.write(r["registros_sem_data"])

        st.markdown("### Resumo da execução")

        if "excel_nome_original" in r:
            st.markdown("**Planilha enviada:**")
            st.code(r["excel_nome_original"])

        st.markdown("**Excel temporário usado no processamento:**")
        st.code(r["excel_localizado"])

        st.markdown("**Modelo de advertência:**")
        st.code(r["modelo_advertencia_localizado"])

        st.markdown("**Modelo de suspensão:**")
        st.code(r["modelo_suspensao_localizado"])

        st.markdown("**Pasta da execução:**")
        st.code(r["pasta_execucao"])

        st.markdown("### Downloads dos PDFs finais")

        if r["pdfs_finais"]:
            for pdf in r["pdfs_finais"]:

                nome_arquivo = os.path.basename(pdf)

                if not os.path.exists(pdf):
                    st.warning(
                        f"Arquivo não encontrado: {nome_arquivo}"
                    )
                    continue

                try:
                    with open(pdf, "rb") as f:
                        pdf_bytes = f.read()

                    st.download_button(
                        label=f"⬇️ Baixar {nome_arquivo}",
                        data=pdf_bytes,
                        file_name=nome_arquivo,
                        mime="application/pdf",
                        key=f"download_{nome_arquivo}"
                    )

                except Exception as erro:
                    st.error(
                        f"Erro ao carregar {nome_arquivo}: {erro}"
                    )

# ============================================
# ABA 2 — ARQUIVOS DETECTADOS
# ============================================

with aba2:
    st.subheader("Arquivos detectados")

    if st.session_state.validacao and "erro" in st.session_state.validacao:
        st.error(st.session_state.validacao["erro"])

    elif st.session_state.validacao:
        v = st.session_state.validacao

        if v.get("excel_nome_original"):
            st.markdown("**Nome original da planilha enviada:**")
            st.code(v["excel_nome_original"])

        st.markdown("**Excel temporário detectado:**")
        st.code(v["excel"])

        st.markdown("**Modelo de advertência detectado:**")
        st.code(v["advertencia"])

        st.markdown("**Modelo de suspensão detectado:**")
        st.code(v["suspensao"])
    else:
        st.info("Clique em Validar para conferir o Excel enviado e os modelos encontrados automaticamente.")

# ============================================
# ABA 3 — LOGS
# ============================================

with aba3:
    st.subheader("Logs")

    if st.session_state.resultado and "logs" in st.session_state.resultado:
        logs_texto = "\n".join(st.session_state.resultado["logs"])
        st.text_area("Saída do processamento", logs_texto, height=420)
    else:
        st.info("Os logs aparecerão aqui após o processamento.")

Writing /content/app_streamlit.py


In [ ]:
!streamlit run /content/app_streamlit.py --server.port 8501 --server.address 0.0.0.0 &>/content/streamlit.log &

In [ ]:
import subprocess
import time
import re

processo_cf = subprocess.Popen(
    [
        "cloudflared",
        "tunnel",
        "--url",
        "http://localhost:8501"
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Gerando link do Cloudflare...")

url = None

for _ in range(60):

    linha = processo_cf.stdout.readline()

    print(linha.strip())

    resultado = re.search(
        r"https://.*?trycloudflare.com",
        linha
    )

    if resultado:
        url = resultado.group(0)
        break

    time.sleep(1)

if url:

    print("\n==================================================")
    print("SITE DISPONÍVEL EM:")
    print(url)
    print("==================================================")

else:

    print(
        "Não foi possível obter o link."
    )

Gerando link do Cloudflare...
2026-06-22T16:38:29Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-06-22T16:38:29Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-06-22T16:38:33Z INF +--------------------------------------------------------------------------------------------+
2026-06-22T16:38:33Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-06-22T16:38:33Z INF |  https://lit-secondary-s

In [ ]:
#!pkill -f streamlit || true
#!pkill -f cloudflared || true
#!fuser -k 8501/tcp || true

In [ ]:
!sed -n '1,200p' /content/streamlit.log



2026-06-22 16:38:34.867 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.226.127:8501

